# IonoAutoML quick visual checks

Plotly notebook for station quality, time normalization checks, and quick data review. No training is performed here.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path("..") if Path("../normalize_time_grid.py").exists() else Path(".")
SCAN_DIR = BASE / "quality_scan_top55_5min"
NORMALIZED_DIR = BASE / "normalized_2024_2025_top3_5min_by_station"
STATION_SET = BASE / "configs" / "station_sets" / "exploration_v0.1.json"
STATION_METADATA = BASE / "configs" / "stations_metadata.csv"
FIG_DIR = BASE / "figures" / "quick_quality"


In [ ]:
PUBLICATION_TEMPLATE = "plotly_white"
FONT_FAMILY = "Arial"
WIDTH = 1100
HEIGHT = 650
EXPORT_FIGURES = False  # Set True only when final PNG/SVG/HTML files are needed.

CLASS_ORDER = ["good", "usable", "weak", "exclude"]
CLASS_COLORS = {
    "good": "#2ca02c",
    "usable": "#ff7f0e",
    "weak": "#9467bd",
    "exclude": "#d62728",
}
CLASS_PATTERNS = {
    "good": "",
    "usable": "/",
    "weak": "x",
    "exclude": ".",
}
LINE_STYLES = {
    "foF2": dict(color="#1F77B4", width=2.4, dash="solid"),
    "Kp": dict(color="#D62728", width=2.0, dash="dash"),
    "Dst": dict(color="#2CA02C", width=2.0, dash="dot"),
}
MARKERS = {
    "good": "circle",
    "usable": "square",
    "weak": "diamond",
    "exclude": "x",
}

def apply_article_layout(fig, title, x_title=None, y_title=None, legend_title=None, height=HEIGHT):
    fig.update_layout(
        template=PUBLICATION_TEMPLATE,
        title=dict(text=title, x=0.02, xanchor="left"),
        font=dict(family=FONT_FAMILY, size=16, color="#111111"),
        width=WIDTH,
        height=height,
        legend=dict(title=legend_title, orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=80, r=40, t=90, b=70),
    )
    if x_title:
        fig.update_xaxes(title_text=x_title, showgrid=True, gridcolor="#E6E6E6", zeroline=False, linecolor="#222222")
    if y_title:
        fig.update_yaxes(title_text=y_title, showgrid=True, gridcolor="#E6E6E6", zeroline=False, linecolor="#222222")
    return fig

def save_figure(fig, name):
    if not EXPORT_FIGURES:
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(FIG_DIR / f"{name}.html")
    try:
        fig.write_image(FIG_DIR / f"{name}.png", scale=2)
        fig.write_image(FIG_DIR / f"{name}.svg")
    except Exception as exc:
        print(f"Static export skipped for {name}: {exc}")


## 1. Station quality summary

In [ ]:
summary_path = SCAN_DIR / "reports" / "station_quality_summary.csv"
summary = pd.read_csv(summary_path)
summary.head()


In [ ]:
counts = summary["quality_class"].value_counts().reindex(CLASS_ORDER, fill_value=0).reset_index()
counts.columns = ["quality_class", "count"]
fig = px.bar(
    counts,
    x="quality_class",
    y="count",
    color="quality_class",
    color_discrete_map=CLASS_COLORS,
    category_orders={"quality_class": CLASS_ORDER},
    text="count",
)
fig.update_traces(textposition="outside")
apply_article_layout(fig, "Station quality classes", "Quality class", "Number of stations", "Quality class", height=560)
fig.show()
save_figure(fig, "station_quality_classes")


In [ ]:
fig = px.scatter(
    summary,
    x="foF2_missing_percent",
    y="foF2_original_percent",
    color="quality_class",
    symbol="quality_class",
    color_discrete_map=CLASS_COLORS,
    symbol_map=MARKERS,
    category_orders={"quality_class": CLASS_ORDER},
    hover_data=["station", "test_original_percent", "foF2_longest_gap_hours", "warnings"],
)
fig.update_traces(marker=dict(size=11, line=dict(width=1.0)))
fig.add_vline(x=40, line_dash="dash", line_color="#d62728", annotation_text="strict missing limit")
fig.add_hline(y=50, line_dash="dot", line_color="#1f77b4", annotation_text="min original coverage")
apply_article_layout(fig, "foF2 coverage by station", "Missing foF2, %", "Original foF2, %", "Quality class")
fig.show()
save_figure(fig, "fof2_coverage_scatter")


## 2. Station set exploration_v0.1

In [ ]:
station_set = json.loads(STATION_SET.read_text(encoding="utf-8"))
stations = station_set["stations"]
print(station_set["station_set_id"], station_set["station_count"])
stations[:20]


## 3. World map of selected stations

Geographic view of the selected exploration stations. Marker color shows the quality class, and marker symbol also changes for print-friendly reading.

In [ ]:
metadata = pd.read_csv(STATION_METADATA)
selected_quality = summary[summary["station"].isin(stations)].copy()
station_map = selected_quality.merge(metadata, on="station", how="left")
missing_coords = station_map[station_map["latitude"].isna() | station_map["longitude_180"].isna()]["station"].tolist()
print(f"Selected stations: {len(stations)}")
print(f"Stations with coordinates: {station_map['latitude'].notna().sum()}")
print(f"Missing coordinates: {missing_coords}")
station_map[["station", "quality_class", "country", "location", "latitude", "longitude_180", "metadata_source"]].sort_values("station").head(12)


In [ ]:
fig = px.scatter_geo(
    station_map,
    lat="latitude",
    lon="longitude_180",
    color="quality_class",
    symbol="quality_class",
    color_discrete_map=CLASS_COLORS,
    symbol_map=MARKERS,
    category_orders={"quality_class": CLASS_ORDER},
    hover_name="station",
    hover_data={
        "country": True,
        "location": True,
        "latitude": ":.2f",
        "longitude_180": ":.2f",
        "foF2_original_percent": ":.1f",
        "foF2_missing_percent": ":.1f",
        "metadata_source": True,
    },
)
fig.update_traces(marker=dict(size=10, line=dict(color="#111111", width=0.7), opacity=0.9))
fig.update_geos(
    projection_type="natural earth",
    showland=True,
    landcolor="#F5F5F5",
    showocean=True,
    oceancolor="#FFFFFF",
    showcountries=True,
    countrycolor="#BDBDBD",
    coastlinecolor="#777777",
    lataxis_showgrid=True,
    lonaxis_showgrid=True,
    lataxis_gridcolor="#DDDDDD",
    lonaxis_gridcolor="#DDDDDD",
)
apply_article_layout(fig, "Selected ionosonde stations on world map", None, None, "Quality class", height=620)
fig.show()
save_figure(fig, "selected_stations_world_map")


In [ ]:
def latitude_band(lat):
    if pd.isna(lat):
        return "missing"
    if lat <= -60:
        return "S high (60-90)"
    if lat < -30:
        return "S mid (30-60)"
    if lat < 30:
        return "Low (-30..30)"
    if lat < 60:
        return "N mid (30-60)"
    return "N high (60-90)"

band_order = ["S high (60-90)", "S mid (30-60)", "Low (-30..30)", "N mid (30-60)", "N high (60-90)"]
station_map["latitude_band"] = station_map["latitude"].apply(latitude_band)
bands = station_map.groupby(["latitude_band", "quality_class"], observed=False).size().reset_index(name="count")
fig = px.bar(
    bands,
    x="latitude_band",
    y="count",
    color="quality_class",
    pattern_shape="quality_class",
    color_discrete_map=CLASS_COLORS,
    pattern_shape_map=CLASS_PATTERNS,
    category_orders={"latitude_band": band_order, "quality_class": CLASS_ORDER},
    text="count",
)
fig.update_traces(textposition="inside")
apply_article_layout(fig, "Selected stations by latitude band", "Latitude band, degrees", "Number of stations", "Quality class", height=520)
fig.show()
save_figure(fig, "selected_stations_latitude_bands")

station_map.groupby("latitude_band").size().reindex(band_order, fill_value=0).reset_index(name="stations")


## 4. One-station time-series plot

For top-3 normalized CSV files, available examples are NI63, PQ052, TR169.

In [ ]:
station = "TR169"
path = NORMALIZED_DIR / "stations" / f"{station}_time_grid.csv"
df = pd.read_csv(path, parse_dates=["time_utc"])
print(df.shape)
df[["time_utc", "foF2", "gfz_Kp", "dst_nT", "had_original_giro_row", "giro_rows_in_interval"]].head()


In [ ]:
plot_df = df.tail(14 * 24 * 12)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=plot_df["time_utc"], y=plot_df["foF2"], name="foF2", mode="lines", line=LINE_STYLES["foF2"]), secondary_y=False)
fig.add_trace(go.Scatter(x=plot_df["time_utc"], y=plot_df["gfz_Kp"], name="Kp", mode="lines", line=LINE_STYLES["Kp"]), secondary_y=True)
fig.add_trace(go.Scatter(x=plot_df["time_utc"], y=plot_df["dst_nT"], name="Dst", mode="lines", line=LINE_STYLES["Dst"]), secondary_y=True)
apply_article_layout(fig, f"{station}: foF2 with Kp and Dst", "Time, UTC", "foF2, MHz", None)
fig.update_yaxes(title_text="Kp / Dst", secondary_y=True, showgrid=False, linecolor="#222222")
fig.show()
save_figure(fig, f"{station}_fof2_kp_dst")


In [ ]:
missing = df.assign(foF2_missing=df["foF2"].isna().astype(int))
plot_missing = missing.tail(30 * 24 * 12)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_missing["time_utc"],
    y=plot_missing["foF2_missing"],
    mode="lines",
    fill="tozeroy",
    name="Missing foF2",
    line=dict(color="#d62728", width=1.8, dash="solid"),
))
apply_article_layout(fig, f"{station}: foF2 missing intervals", "Time, UTC", "Missing flag", None, height=420)
fig.update_yaxes(tickvals=[0, 1], ticktext=["observed", "missing"])
fig.show()
save_figure(fig, f"{station}_fof2_missing_intervals")


## 5. Time-step comparison

In [ ]:
scan_dirs = {
    "5min": BASE / "quality_scan_exploration_v0_1_5min",
    "15min": BASE / "quality_scan_exploration_v0_1_15min",
    "30min": BASE / "quality_scan_exploration_v0_1_30min",
    "60min": BASE / "quality_scan_exploration_v0_1_60min",
}
rows = []
for step, directory in scan_dirs.items():
    path = directory / "reports" / "station_quality_summary.csv"
    if not path.exists():
        continue
    s = pd.read_csv(path)
    rows.append({
        "step": step,
        "stations": len(s),
        "good": int((s["quality_class"] == "good").sum()),
        "usable": int((s["quality_class"] == "usable").sum()),
        "weak": int((s["quality_class"] == "weak").sum()),
        "exclude": int((s["quality_class"] == "exclude").sum()),
        "median_original": s["foF2_original_percent"].median(),
        "median_missing": s["foF2_missing_percent"].median(),
    })
compare = pd.DataFrame(rows)
compare


In [ ]:
if not compare.empty:
    long = compare.melt(id_vars="step", value_vars=CLASS_ORDER, var_name="quality_class", value_name="count")
    fig = px.bar(
        long,
        x="step",
        y="count",
        color="quality_class",
            color_discrete_map=CLASS_COLORS,
            category_orders={"quality_class": CLASS_ORDER},
        barmode="stack",
        text="count",
    )
    fig.update_traces(textposition="inside")
    apply_article_layout(fig, "Quality classes by time step", "Time step", "Number of stations", "Quality class", height=560)
    fig.show()
    save_figure(fig, "quality_classes_by_time_step")
